In [ ]:
# churn_pipeline.py
import os
import json
import joblib
import yaml
import shap
import lightgbm as lgb
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss


In [ ]:
# ---------------- CONFIGURATION ----------------
CONFIG = {
    "W_days": 90,
    "H_days": 30,
    "min_rows": 1000,
    "storage_root": "models_storage",
    "data_root": "data",
    "model_params": {
        "objective": "binary",
        "learning_rate": 0.05,
        "num_leaves": 64,
        "n_estimators": 800,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1
    }
}

In [ ]:
# ---------------- BASIC HELPERS ----------------
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def safe_log(msg):
    print(f"[INFO] {msg}")

def evaluate_scores(y_true, p):
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p))
    }

def psi(expected, actual, bins=10):
    e, edges = np.histogram(expected, bins=bins)
    a, _ = np.histogram(actual, bins=edges)
    e = e / (e.sum() + 1e-9)
    a = a / (a.sum() + 1e-9)
    return float(np.sum((a - e) * np.log((a + 1e-9) / (e + 1e-9))))



In [ ]:
# ---------------- FEATURE ENGINEERING ----------------
def build_snapshots(events, W_days, H_days):
    df = events.copy()
    df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)
    max_t = df["event_timestamp"].max().normalize()
    t_refs = pd.date_range(end=max_t, periods=6, freq="30D")

    frames = []
    for t_ref in t_refs:
        W_start = t_ref - timedelta(days=W_days)
        H_end = t_ref + timedelta(days=H_days)
        hist = df[(df["event_timestamp"] > W_start) & (df["event_timestamp"] <= t_ref)]
        future = df[(df["event_timestamp"] > t_ref) & (df["event_timestamp"] <= H_end)]

        agg = hist.groupby("customer_id").agg(
            events_90d=("event_timestamp", "count"),
            spend_90d=("amount", "sum"),
            avg_spend=("amount", "mean"),
            last_event=("event_timestamp", "max")
        ).reset_index()

        agg["recency_days"] = (t_ref - agg["last_event"]).dt.days
        agg["tenure_days"] = (t_ref - df.groupby("customer_id")["event_timestamp"].transform("min").fillna(t_ref)).dt.days.groupby(df["customer_id"]).max()

        active_future = future.groupby("customer_id")["event_timestamp"].count().rename("future_events")
        feats = agg.merge(active_future, on="customer_id", how="left").fillna({"future_events": 0})
        feats["label"] = (feats["future_events"] == 0).astype(int)
        feats["t_ref"] = t_ref
        frames.append(feats.drop(columns=["last_event", "future_events"]))

    df_final = pd.concat(frames, ignore_index=True).fillna(0)
    return df_final


In [ ]:
# ---------------- MODEL TRAINING ----------------
def train_user_model(user_id):
    safe_log(f"=== Training model for user: {user_id} ===")
    data_dir = os.path.join(CONFIG["data_root"], user_id)
    files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".csv") or f.endswith(".parquet")]
    if not files:
        raise FileNotFoundError("No user dataset found.")

    df = pd.concat([pd.read_csv(f) if f.endswith(".csv") else pd.read_parquet(f) for f in files])
    if "customer_id" not in df.columns or "timestamp" not in df.columns:
        raise ValueError("Dataset must contain 'customer_id' and 'timestamp' columns.")

    df = df.rename(columns={"timestamp": "event_timestamp"})
    if "amount" not in df.columns:
        df["amount"] = 0.0

    feats = build_snapshots(df, CONFIG["W_days"], CONFIG["H_days"])
    if feats["label"].nunique() < 2:
        raise ValueError("Not enough churn/no-churn variation.")

    X = feats.drop(columns=["label", "t_ref", "customer_id"])
    y = feats["label"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)
    scale_pos_weight = (1 - y_train.mean()) / (y_train.mean() + 1e-9)

    model = lgb.LGBMClassifier(**CONFIG["model_params"])
    weights = compute_sample_weight(class_weight={0:1.0, 1:scale_pos_weight}, y=y_train)
    model.fit(X_train, y_train, sample_weight=weights)

    p_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_raw, y_val)
    p_cal = iso.transform(p_raw)
    metrics = evaluate_scores(y_val, p_cal)

    # Save artifacts
    model_dir = ensure_dir(os.path.join(CONFIG["storage_root"], user_id))
    version = len([d for d in os.listdir(model_dir) if d.startswith("v")]) + 1
    version_dir = ensure_dir(os.path.join(model_dir, f"v{version}"))

    joblib.dump(model, os.path.join(version_dir, "model.pkl"))
    joblib.dump(iso, os.path.join(version_dir, "iso.pkl"))
    json.dump({
        "user_id": user_id,
        "features": list(X.columns),
        "metrics": metrics,
        "version": version
    }, open(os.path.join(version_dir, "metadata.json"), "w"))

    safe_log(f"✅ Model trained for {user_id} | PR-AUC={metrics['pr_auc']:.3f} | saved v{version}")
    return metrics, version



In [ ]:
# ---------------- PREDICTION ----------------
def predict_user(user_id, input_df):
    model_dir = os.path.join(CONFIG["storage_root"], user_id)
    versions = sorted([d for d in os.listdir(model_dir) if d.startswith("v")])
    if not versions:
        raise FileNotFoundError("No model found for this user.")
    latest = versions[-1]
    model_path = os.path.join(model_dir, latest, "model.pkl")
    iso_path = os.path.join(model_dir, latest, "iso.pkl")
    meta = json.load(open(os.path.join(model_dir, latest, "metadata.json")))

    model = joblib.load(model_path)
    iso = joblib.load(iso_path)
    feat_cols = meta["features"]

    for f in feat_cols:
        if f not in input_df:
            input_df[f] = 0
    X = input_df[feat_cols]
    preds_raw = model.predict_proba(X)[:, 1]
    preds = iso.transform(preds_raw)

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    top_reasons = []
    for row in shap_vals:
        idx = np.argsort(-np.abs(row))[:3]
        top_reasons.append([f"{feat_cols[i]} ({row[i]:.3f})" for i in idx])

    return preds, top_reasons



In [ ]:
 ---------------- RETRAIN ----------------
def retrain_user(user_id):
    safe_log(f"🔁 Retraining model for {user_id}")
    return train_user_model(user_id)


# ---------------- DRIFT DETECTION ----------------
def check_drift(user_id):
    model_dir = os.path.join(CONFIG["storage_root"], user_id)
    versions = sorted([d for d in os.listdir(model_dir) if d.startswith("v")])
    if len(versions) < 2:
        safe_log("Only one version found, drift check skipped.")
        return None
    v_prev, v_curr = versions[-2], versions[-1]
    prev_meta = json.load(open(os.path.join(model_dir, v_prev, "metadata.json")))
    curr_meta = json.load(open(os.path.join(model_dir, v_curr, "metadata.json")))
    drift = {}
    for m in ["pr_auc", "roc_auc", "brier"]:
        drift[m] = curr_meta["metrics"][m] - prev_meta["metrics"][m]
    safe_log(f"📈 Drift check between {v_prev}→{v_curr}: {drift}")
    return drift


In [ ]:
# ---------------- EXAMPLE RUN ----------------
if __name__ == "__main__":
    user = "user123"

    # 1️⃣ Train model
    metrics, version = train_user_model(user)

    # 2️⃣ Predict on new data
    new_data = pd.DataFrame([
        {"events_90d": 4, "spend_90d": 200, "avg_spend": 50, "recency_days": 5, "tenure_days": 60},
        {"events_90d": 0, "spend_90d": 0, "avg_spend": 0, "recency_days": 100, "tenure_days": 15}
    ])
    scores, reasons = predict_user(user, new_data)
    print("\nPredictions:\n", scores)
    print("Reasons:\n", reasons)

    # 3️⃣ Retrain later
    retrain_user(user)

    # 4️⃣ Check drift between versions
    check_drift(user)